# Experiment No. 3 – Data Connection and Analysis using Advanced Visualizations

**Aim:** To perform data connection and analysis by applying various plots and visualization techniques on numerical and categorical data to identify patterns, relationships, and insights.

**Dataset:** Cleaned *Data Science Jobs in India (Naukri.com)* dataset produced in Experiment 2 (`cleaned_ds_jobs_india.csv`).
Original source: https://www.kaggle.com/datasets/anandhuh/data-science-jobs-in-india

**Advanced plots used:**
1. Violin plot with swarm overlay – numerical vs categorical
2. Radar (spider) chart – multivariate categorical profile
3. Sankey diagram – flow between categorical variables
4. 3D bar chart – two categories and a count in one view

## Step 1 – Connect to the Data

In [ ]:
# Import the required libraries
import os                               # to build file paths
import numpy as np                      # for numeric arrays and angles
import pandas as pd                     # to load and handle the data table
import matplotlib.pyplot as plt         # to draw charts
import seaborn as sns                   # to draw statistical charts easily
import plotly.graph_objects as go       # for the interactive Sankey diagram

# Use a white background with grid lines for all charts
sns.set_theme(style='whitegrid')

Run **only one** of the next two cells: Option 1 on a laptop, Option 2 on Google Colab.

In [ ]:
# OPTION 1 - Running on laptop (CSV file is in the same folder as this notebook)
path = '.'    # '.' means the current folder

In [ ]:
# OPTION 2 - Running on Google Colab (upload cleaned_ds_jobs_india.csv first)
# Remove the # from the two lines below when using Colab.

# from google.colab import files
# files.upload(); path = '.'

In [ ]:
# Read the cleaned CSV file (output of Experiment 2) into a DataFrame
df = pd.read_csv(os.path.join(path, 'cleaned_ds_jobs_india.csv'))

print('Dataset shape:', df.shape)
df.head()

In [ ]:
# Split the columns into numerical and categorical types
numerical_cols   = df.select_dtypes(include='number').columns.tolist()
categorical_cols = ['Role', 'Main_City', 'Experience_Level']
skill_cols       = [c for c in df.columns if c.startswith('Needs_')]   # True/False skill flags

print('Numerical columns  :', numerical_cols)
print('Categorical columns:', categorical_cols)
print('Skill flag columns :', skill_cols)

In [ ]:
# Summary of the numerical columns
df[numerical_cols].describe().round(2)

## Step 2 – Prepare the Data for Plotting

- The **Other** role is a mixed group, so it is left out to keep the comparisons clear.
- Only the **top 6 cities** are kept for the city charts; the rest have very few jobs.
- Experience levels are put in their natural order (Fresher → Senior).

In [ ]:
# Keep the five main roles (drop the mixed 'Other' group)
main = df[df['Role'] != 'Other'].copy()

# Order roles by their median experience, from lowest to highest
role_order = main.groupby('Role')['Avg_Exp'].median().sort_values().index.tolist()

# Natural order for the experience levels
level_order = ['Fresher (0-1)', 'Junior (2-3)', 'Mid (4-6)', 'Senior (7+)']

# Top 6 cities by number of jobs
top_cities = main['Main_City'].value_counts().head(6).index.tolist()

print('Rows used    :', len(main))
print('Role order   :', role_order)
print('Top 6 cities :', top_cities)

## Step 3 – Advanced Visualizations

### Plot 1 – Violin Plot with Swarm Overlay (Average Experience by Role)

A **violin plot** shows the full shape of a numerical distribution for each category: wide parts mean many jobs at that experience value. The **swarm points** on top show a random sample of real jobs, so the spread of individual postings is visible.

In [ ]:
plt.figure(figsize=(12, 6))

# Violin: full distribution of Avg_Exp for each role (inner box shows median and quartiles)
sns.violinplot(data=main, x='Role', y='Avg_Exp', order=role_order,
               hue='Role', palette='Set2', inner='box', cut=0, legend=False)

# Swarm: 600 randomly sampled jobs drawn as dots (all 10,000+ would hide the violins)
sample = main.sample(600, random_state=42)
sns.swarmplot(data=sample, x='Role', y='Avg_Exp', order=role_order,
              color='black', size=2.5, alpha=0.6)

plt.title('Distribution of Average Experience Required by Role')
plt.xlabel('Role')
plt.ylabel('Average Experience Required (years)')
plt.ylim(0, 20)
plt.show()

# Median and mean experience per role, to support the chart
main.groupby('Role')['Avg_Exp'].agg(['median', 'mean', 'count']).loc[role_order].round(2)

**Observation:** **Data / Business Analyst** jobs ask for the least experience (median **4.5 years**), and their violin is widest between 3 and 5 years, so this is the most entry-friendly role. **Data Engineer** has the highest median (**6.0 years**). Data Scientist, ML / AI Engineer and Software Developer sit in between (median 5.5 years). Every role has a long thin upper tail, which comes from a few senior posts asking for 15+ years. The ML / AI Engineer violin is uneven because that role has only 212 jobs.

### Plot 2 – Radar (Spider) Chart (Skill Profile of Each Role)

Each axis of a **radar chart** is one skill, and the distance from the centre is the **percentage of jobs in that role asking for the skill**. Every role becomes a closed shape, so the skill pattern of the roles can be compared at a glance.

In [ ]:
# Percentage of jobs in each role that need each skill
skill_pct = main.groupby('Role')[skill_cols].mean() * 100
skill_pct.columns = [c.replace('Needs_', '').replace('_', ' ') for c in skill_cols]
skill_pct = skill_pct.loc[role_order]
skill_pct.round(1)

In [ ]:
labels = skill_pct.columns.tolist()
n = len(labels)

# One angle per skill; repeat the first angle at the end to close each shape
angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors = sns.color_palette('Set2', len(skill_pct))

for (role, row), color in zip(skill_pct.iterrows(), colors):
    values = row.tolist() + row.tolist()[:1]           # close the shape
    ax.plot(angles, values, color=color, linewidth=2, label=role)
    ax.fill(angles, values, color=color, alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=11)
ax.set_rlabel_position(90)
ax.set_ylim(0, 80)
ax.set_yticks([20, 40, 60, 80])
ax.set_yticklabels(['20%', '40%', '60%', '80%'], fontsize=8)
ax.set_title('Skill Demand Profile by Role (% of jobs needing each skill)', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1))
plt.show()

**Observation:** The chart shows a clear pattern for each role. **ML / AI Engineer (77.8%) and Data Scientist (57.0%)** shapes stretch towards **Machine Learning**. **Python** is shared by all technical roles (30–44%) but is asked for in only 10.9% of analyst jobs. The **Data / Business Analyst** shape is small but leans towards **Excel (11.7%), Tableau (9.7%) and Power BI (7.6%)**, tools that almost no other role asks for (Excel under 1%). **Data Engineer is the only role with real AWS demand (15.5%)**, which shows its cloud focus.

### Plot 3 – Sankey Diagram (Experience Level → Role → City)

A **Sankey diagram** shows how records *flow* from one category to the next. The width of each band is the number of jobs. Here every job flows from its experience level, to its role, to its city, which connects three categorical columns in one chart.

In [ ]:
# Group cities outside the top 6 as 'Other Cities'
flow = main.copy()
flow['City'] = flow['Main_City'].where(flow['Main_City'].isin(top_cities), 'Other Cities')
city_order = top_cities + ['Other Cities']

# Node list: experience levels, then roles, then cities
nodes = level_order + role_order + city_order
idx = {name: i for i, name in enumerate(nodes)}

# Links: Experience_Level -> Role, and Role -> City (with job counts)
lr = flow.groupby(['Experience_Level', 'Role']).size().reset_index(name='jobs')
rc = flow.groupby(['Role', 'City']).size().reset_index(name='jobs')

source = [idx[s] for s in lr['Experience_Level']] + [idx[s] for s in rc['Role']]
target = [idx[t] for t in lr['Role']]             + [idx[t] for t in rc['City']]
value  = lr['jobs'].tolist() + rc['jobs'].tolist()

# Colour each band by its role so the flow of a role can be followed
role_rgb = {r: c for r, c in zip(role_order, sns.color_palette('Set2', len(role_order)).as_hex())}
def rgba(hex_color, a=0.45):
    h = hex_color.lstrip('#')
    return f'rgba({int(h[0:2],16)},{int(h[2:4],16)},{int(h[4:6],16)},{a})'
link_colors = [rgba(role_rgb[r]) for r in lr['Role']] + [rgba(role_rgb[r]) for r in rc['Role']]
node_colors = (['#8c8c8c'] * len(level_order) + [role_rgb[r] for r in role_order]
               + ['#5b7fa6'] * len(city_order))

fig = go.Figure(go.Sankey(
    node=dict(label=nodes, color=node_colors, pad=18, thickness=18),
    link=dict(source=source, target=target, value=value, color=link_colors)))
fig.update_layout(title='Flow of Jobs: Experience Level → Role → City',
                  font_size=12, width=1000, height=600)
fig.show()

In [ ]:
# Supporting table: share of each experience level within each role (%)
pd.crosstab(main['Role'], main['Experience_Level'], normalize='index')[level_order].loc[role_order].mul(100).round(1)

**Observation:** The **Junior (2-3) and Mid (4-6)** bands are the thickest, so most jobs need 2–6 years of experience. The **Fresher** band flows mostly into **Data / Business Analyst (868 of 1,374 fresher jobs, about 63%)**, while the **Senior** band flows mostly into **Data Engineer (598 of 1,298, about 46%)**. On the right side, **Bengaluru** gets the widest band from every role and holds about 31% of all these jobs.

### Plot 4 – 3D Bar Chart (Jobs by City and Role)

A **3D bar chart** places two categorical variables on the X and Y axes and the count as the **height** of each bar. This shows all city–role combinations together, so the biggest job markets stand out.

In [ ]:
# Job counts for every (city, role) pair, top 6 cities only
counts = (main[main['Main_City'].isin(top_cities)]
          .groupby(['Main_City', 'Role']).size()
          .unstack(fill_value=0)
          .loc[top_cities, role_order])
counts

In [ ]:
fig = plt.figure(figsize=(13, 9))
ax = fig.add_subplot(111, projection='3d')

colors = sns.color_palette('Set2', len(role_order))
dx, dy = 0.5, 0.5                                   # width and depth of each bar

for j, role in enumerate(role_order):               # Y axis: role
    for i, city in enumerate(top_cities):           # X axis: city
        ax.bar3d(i, j, 0, dx, dy, counts.loc[city, role],
                 color=colors[j], alpha=0.85, shade=True)

ax.set_xticks(np.arange(len(top_cities)) + dx / 2)
ax.set_xticklabels(top_cities, rotation=20, ha='right')
ax.set_yticks(np.arange(len(role_order)) + dy / 2)
ax.set_yticklabels(role_order, rotation=-15, ha='left', fontsize=9)
ax.set_zlabel('Number of Jobs', labelpad=8)
ax.set_title('Number of Jobs by City and Role (3D Bar Chart)', pad=10)
ax.view_init(elev=25, azim=-55)                     # camera angle
plt.tight_layout()
plt.show()

**Observation:** **Bengaluru has the tallest bar for every role** (1,162 analyst and 1,095 data engineer jobs). The 3D view also shows that cities have different specialities. **Mumbai and Gurugram lean towards analyst jobs** (605 vs 254 and 431 vs 257 data engineer jobs), while **Pune and Hyderabad lean towards Data Engineer** (696 vs 322 and 647 vs 450). ML / AI Engineer bars are tiny everywhere; only Bengaluru crosses 100 jobs.

## Conclusion

In this experiment, the cleaned job dataset from Experiment 2 was connected and analysed using four advanced visualizations. Each plot handled a different kind of data:

- The **violin + swarm plot** compared a numerical column (experience) across categories and showed that analyst roles are the most entry-friendly, while data engineering asks for the most experience.
- The **radar chart** compared seven skill flags at once and showed a clear skill pattern for each role: ML for scientists and ML engineers, BI tools for analysts and AWS for data engineers.
- The **Sankey diagram** connected three categorical columns and showed how fresher jobs flow mostly into analyst roles and how Bengaluru gets the largest share of every role.
- The **3D bar chart** showed two categories and a count together and revealed city specialities: Mumbai and Gurugram for analysts, Pune and Hyderabad for data engineers.
These advanced plots revealed patterns and relationships that simple bar or pie charts of one column at a time would not show.